# Day 03：损失函数与梯度下降 —— 模型的「吐槽大会」

> ☀️ 第二周 · 破局与复兴 · 第 3 天

昨天我们学了激活函数——让神经元有了「非线性」的能力。

但光有能力还不够，模型还需要知道两件事：
1. **自己错得有多离谱**（损失函数）
2. **怎么改正错误**（梯度下降）

今天把这两个核心问题讲清楚。

**今天的任务**：理解不同损失函数的适用场景，掌握梯度下降的核心思想。

---
## 1. 历史背景：从「拍脑袋」到「算账」

### 早期的困境

1943 年 McCulloch-Pitts 神经元诞生后，人们面临一个尴尬的问题：**怎么知道模型学得好不好？**

早期的方法很粗暴——人工检查输出，手动调整权重。就像没有秤的厨师，全凭手感加盐。

### 损失函数的诞生

1986 年，Rumelhart 等人在反向传播论文中明确了**损失函数（Loss Function）**的概念：用一个数学公式来量化「预测和真实之间的差距」。

有了损失函数，模型就有了明确的目标——**最小化损失**。

### 梯度下降的渊源

梯度下降的思想可以追溯到 1847 年，数学家 Augustin-Louis Cauchy 提出了用梯度方向求解函数最小值的方法。

但直到计算机普及后，梯度下降才真正成为训练神经网络的核心工具。它的本质很简单：**在山上找最陡的下坡路，一步步走到谷底**。

---
## 2. 生活隐喻：GPS 寻路

### 损失函数 = GPS 的「距离提示」

想象你开车去一个陌生地方，GPS 告诉你「距离目的地还有 5 公里」。这个「5 公里」就是**损失**——你当前位置和目标之间的差距。

不同的损失函数就像不同的测量方式：
- **MSE（均方误差）**：直线距离的平方。距离越远，惩罚越重（5 公里的误差比 1 公里严重 25 倍）
- **MAE（平均绝对误差）**：直线距离。距离多远就罚多少，线性关系
- **BCE（交叉熵）**：「你走错方向的概率有多大」。专门用于分类问题

### 梯度下降 = 盲人下山

想象你蒙着眼睛站在山上，目标是走到最低点。你的策略是：
1. 用脚感受脚下的坡度（计算梯度）
2. 往最陡的下坡方向迈一步（沿梯度反方向更新）
3. 重复，直到感觉脚下是平的（收敛）

**学习率**就是你每步迈多远：
- 步子太大：可能跨过最低点，甚至走到更高的地方（发散）
- 步子太小：走到天黑也到不了（收敛太慢）
- 步子刚好：稳稳当当地走到谷底

---
## 3. 数学直觉：损失函数

### 为什么需要损失函数？

损失函数是模型和现实之间的「翻译器」。模型输出一个数字，损失函数告诉它「这个数字离正确答案有多远」。

没有损失函数，模型就像没有靶子的射箭手——不知道往哪射，也不知道射得准不准。

### MSE（均方误差）

$$L_{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

- 把每个预测误差平方后求平均
- **特点**：对大误差惩罚很重（平方效应）。预测差 2 是差 1 的 4 倍惩罚
- **适用**：回归任务（预测房价、温度等连续值）

### MAE（平均绝对误差）

$$L_{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

- 把每个预测误差取绝对值后求平均
- **特点**：对大误差惩罚较轻（线性关系）。预测差 2 只是差 1 的 2 倍惩罚
- **适用**：回归任务，尤其当数据有异常值时（不会被极端值主导）

### BCE（二元交叉熵）

$$L_{BCE} = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i)]$$

- 专门用于二分类任务（输出是 0 或 1 的概率）
- **特点**：当预测偏离真实越远，惩罚呈指数增长
- **适用**：二分类任务（判断猫/狗、垃圾邮件/正常邮件）

用代码可视化这三种损失函数的形状：

In [ ]:
import torch
import matplotlib.pyplot as plt

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 假设真实值为 1.0
y_true = torch.tensor(1.0)  # 真实标签
y_pred_range = torch.linspace(0.01, 1.99, 200)  # 预测值从 0.01 到 1.99

# MSE 损失：(预测 - 真实)²
mse_values = (y_pred_range - y_true) ** 2

# MAE 损失：|预测 - 真实|
mae_values = torch.abs(y_pred_range - y_true)

# BCE 损失：-[y*log(ŷ) + (1-y)*log(1-ŷ)]
eps = 1e-7  # 防止 log(0)
y_clamped = torch.clamp(y_pred_range, eps, 1 - eps)  # 限制在 (0,1) 范围
bce_values = -(y_true * torch.log(y_clamped) + (1 - y_true) * torch.log(1 - y_clamped))

# 画图
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# MSE
axes[0].plot(y_pred_range.numpy(), mse_values.numpy(), 'b-', linewidth=2)
axes[0].axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='真实值=1')
axes[0].set_xlabel('预测值')
axes[0].set_ylabel('MSE 损失')
axes[0].set_title('MSE：对大误差惩罚很重（平方效应）')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(y_pred_range.numpy(), mae_values.numpy(), 'g-', linewidth=2)
axes[1].axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='真实值=1')
axes[1].set_xlabel('预测值')
axes[1].set_ylabel('MAE 损失')
axes[1].set_title('MAE：线性惩罚，对异常值鲁棒')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# BCE
axes[2].plot(y_pred_range.numpy(), bce_values.numpy(), 'purple', linewidth=2)
axes[2].axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='真实值=1')
axes[2].set_xlabel('预测值')
axes[2].set_ylabel('BCE 损失')
axes[2].set_title('BCE：偏离越远惩罚越狠（指数增长）')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_functions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("三种损失函数对比：")
print(f"  MSE  公式: (y - ŷ)²            适用: 回归任务")
print(f"  MAE  公式: |y - ŷ|             适用: 回归（有异常值时）")
print(f"  BCE  公式: -[y·log(ŷ)+(1-y)·log(1-ŷ)]  适用: 二分类")

---
## 4. 数学直觉：梯度下降

### 核心公式

梯度下降的更新规则只有一行：

$$W_{new} = W_{old} - \eta \cdot \frac{\partial L}{\partial W}$$

其中：
- $W$ 是模型的权重参数
- $\eta$（eta）是**学习率**，控制每步走多远
- $\frac{\partial L}{\partial W}$ 是**梯度**，表示「W 增加一点点时，L 会怎么变」

为什么是**减去**梯度？因为梯度指向函数增长最快的方向，我们要最小化损失，所以往反方向走。

### 用一个简单函数演示

假设损失函数是 $f(x) = (x-3)^2 + 1$，最小值在 $x=3$。

梯度是 $f'(x) = 2(x-3)$。

从 $x=0$ 出发，用梯度下降找最小值：

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 定义函数和梯度
def f(x):
    """损失函数：f(x) = (x-3)² + 1，最小值在 x=3"""
    return (x - 3) ** 2 + 1

def grad_f(x):
    """梯度：f'(x) = 2(x-3)"""
    return 2 * (x - 3)

# 梯度下降过程
x = 0.0  # 起始位置
lr = 0.1  # 学习率
trajectory = [x]  # 记录每步的位置

for step in range(20):  # 迭代 20 步
    gradient = grad_f(x)  # 计算当前梯度
    x = x - lr * gradient  # 沿梯度反方向更新
    trajectory.append(x)  # 记录新位置

# 可视化
x_range = np.linspace(-1, 6, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图：函数曲线 + 梯度下降路径
axes[0].plot(x_range, [f(xi) for xi in x_range], 'b-', linewidth=2, label='f(x) = (x-3)² + 1')
axes[0].plot(trajectory, [f(xi) for xi in trajectory], 'ro-', markersize=6, linewidth=1.5, label='梯度下降路径')
axes[0].scatter([3], [1], c='green', s=200, marker='*', zorder=5, label='最小值 x=3')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('梯度下降：一步步走向最小值')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图：损失收敛曲线
losses = [f(xi) for xi in trajectory]
axes[1].plot(range(len(losses)), losses, 'bo-', markersize=6)
axes[1].set_xlabel('迭代步数')
axes[1].set_ylabel('损失 f(x)')
axes[1].set_title('损失随迭代下降')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gradient_descent.png', dpi=150, bbox_inches='tight')
plt.show()

# 打印前几步的过程
print("梯度下降过程（前 8 步）：")
for i in range(8):
    x_val = trajectory[i]
    print(f"  第 {i} 步: x = {x_val:.4f}, f(x) = {f(x_val):.4f}, 梯度 = {grad_f(x_val):.4f}")
print(f"  ...\n  第 20 步: x = {trajectory[-1]:.4f}（接近最小值 3.0）")

### 学习率的影响

学习率 $\eta$ 是梯度下降中最重要的超参数。

用同一个函数，对比不同学习率的效果：

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

def f(x):
    """f(x) = (x-3)² + 1"""
    return (x - 3) ** 2 + 1

def grad_f(x):
    """f'(x) = 2(x-3)"""
    return 2 * (x - 3)

# 不同学习率
learning_rates = [0.01, 0.1, 0.5, 0.9, 1.5]
colors = ['blue', 'green', 'orange', 'red', 'purple']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：不同学习率的下降路径
x_range = np.linspace(-1, 6, 200)
axes[0].plot(x_range, [f(xi) for xi in x_range], 'k-', linewidth=2)
axes[0].scatter([3], [1], c='green', s=200, marker='*', zorder=5)

for lr, color in zip(learning_rates, colors):
    x = 0.0  # 每次从 x=0 出发
    traj = [x]
    for _ in range(30):  # 30 步
        x = x - lr * grad_f(x)
        traj.append(x)
    axes[0].plot(traj, [f(xi) for xi in traj], 'o-', color=color, alpha=0.7,
                 markersize=3, linewidth=1.5, label=f'学习率={lr}')

axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('不同学习率的梯度下降路径')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图：损失收敛曲线对比
for lr, color in zip(learning_rates, colors):
    x = 0.0
    losses = [f(x)]
    for _ in range(30):
        x = x - lr * grad_f(x)
        losses.append(f(x))
    axes[1].plot(range(len(losses)), losses, 'o-', color=color, alpha=0.7,
                 markersize=3, linewidth=1.5, label=f'学习率={lr}')

axes[1].set_xlabel('迭代步数')
axes[1].set_ylabel('损失')
axes[1].set_title('损失收敛曲线对比')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_rate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("学习率的影响：")
print("  0.01 → 太小，收敛很慢，30 步还没到")
print("  0.1  → 刚好，稳定收敛")
print("  0.5  → 较快，但接近最小值时会震荡")
print("  0.9  → 震荡明显，但最终收敛")
print("  1.5  → 太大，发散了！越走越远！")

### 优化器：从 SGD 到 Adam

基础的梯度下降（SGD）有时会遇到问题：在「山谷」里来回震荡，收敛慢。

为了解决这个问题，研究者提出了各种改进：

| 优化器 | 核心思想 | 类比 |
|--------|----------|------|
| SGD | 基础梯度下降 | 纯靠脚感下山 |
| SGD + Momentum | 加入「惯性」 | 推着球下山，球有惯性不会轻易反弹 |
| Adam | 自适应学习率 + 动量 | 自动调整步幅，平路大步走，陡坡小步走 |

**Adam**（Adaptive Moment Estimation）是目前最常用的优化器，它结合了 Momentum（动量）和 RMSProp（自适应学习率）的优点。

简单来说：
- **Momentum**：记住之前的移动方向，减少震荡
- **自适应学习率**：对不同参数自动调整学习率

实际使用时，**Adam 几乎总是默认首选**。

---
## 5. 代码实现：完整训练流程

现在把损失函数和梯度下降结合起来，用一个完整的训练流程解决 XOR 问题。

训练循环的五步：
1. **前向传播**：计算预测值
2. **计算损失**：用损失函数衡量误差
3. **清零梯度**：PyTorch 默认累加梯度，需要手动清零
4. **反向传播**：计算每个参数的梯度
5. **更新参数**：沿梯度反方向调整权重

In [ ]:
import torch

def sigmoid(z):
    """Sigmoid 激活函数"""
    return 1 / (1 + torch.exp(-z))

# XOR 数据
X = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# 手动定义两层网络的权重（requires_grad=True 让 PyTorch 追踪梯度）
W1 = torch.randn(2, 4, requires_grad=True)  # 隐藏层权重：2输入 → 4隐藏
b1 = torch.randn(1, 4, requires_grad=True)  # 隐藏层偏置
W2 = torch.randn(4, 1, requires_grad=True)  # 输出层权重：4隐藏 → 1输出
b2 = torch.randn(1, 1, requires_grad=True)  # 输出层偏置

# 训练参数
lr = 0.5  # 学习率
epochs = 1000  # 训练轮数

print("开始训练 XOR...")
print("=" * 50)

losses = []  # 记录每轮损失

for epoch in range(epochs):
    # 第 1 步：前向传播
    z1 = torch.matmul(X, W1) + b1  # 隐藏层线性变换
    a1 = sigmoid(z1)  # 隐藏层激活
    z2 = torch.matmul(a1, W2) + b2  # 输出层线性变换
    y_pred = sigmoid(z2)  # 输出层激活

    # 第 2 步：计算损失（MSE）
    loss = torch.mean((y_pred - y) ** 2)
    losses.append(loss.item())

    # 第 3 步：清零梯度（PyTorch 默认累加）
    if W1.grad is not None:
        W1.grad.zero_()
        b1.grad.zero_()
        W2.grad.zero_()
        b2.grad.zero_()

    # 第 4 步：反向传播（自动计算所有梯度）
    loss.backward()

    # 第 5 步：更新参数（梯度下降）
    with torch.no_grad():  # 更新参数时不需要追踪梯度
        W1 -= lr * W1.grad
        b1 -= lr * b1.grad
        W2 -= lr * W2.grad
        b2 -= lr * b2.grad

    # 每 200 轮打印一次
    if (epoch + 1) % 200 == 0:
        print(f"  第 {epoch+1} 轮：损失 = {loss.item():.4f}")

print(f"\n训练完成！最终损失 = {losses[-1]:.4f}")

---
## 6. 验证实验

训练完了，验证一下模型是否真的学会了 XOR。

In [ ]:
import torch

def sigmoid(z):
    """Sigmoid 激活函数"""
    return 1 / (1 + torch.exp(-z))

# XOR 数据
X = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# 初始化权重
W1 = torch.randn(2, 4, requires_grad=True)
b1 = torch.randn(1, 4, requires_grad=True)
W2 = torch.randn(4, 1, requires_grad=True)
b2 = torch.randn(1, 1, requires_grad=True)

# 训练
lr = 0.5
for epoch in range(1000):
    z1 = torch.matmul(X, W1) + b1
    a1 = sigmoid(z1)
    z2 = torch.matmul(a1, W2) + b2
    y_pred = sigmoid(z2)
    loss = torch.mean((y_pred - y) ** 2)

    if W1.grad is not None:
        W1.grad.zero_()
        b1.grad.zero_()
        W2.grad.zero_()
        b2.grad.zero_()

    loss.backward()

    with torch.no_grad():
        W1 -= lr * W1.grad
        b1 -= lr * b1.grad
        W2 -= lr * W2.grad
        b2 -= lr * b2.grad

# 验证
with torch.no_grad():
    z1 = torch.matmul(X, W1) + b1
    a1 = sigmoid(z1)
    z2 = torch.matmul(a1, W2) + b2
    y_pred = sigmoid(z2)

print("XOR 训练结果验证")
print("=" * 50)
print(f"{'输入':<12} {'真实标签':<10} {'预测概率':<12} {'预测值':<8} {'结果'}")
print("-" * 50)

all_correct = True
for i in range(4):
    x_str = f"({X[i,0].item()}, {X[i,1].item()})"
    true_label = int(y[i].item())
    prob = y_pred[i].item()
    pred = 1 if prob > 0.5 else 0
    status = "✓ 正确" if true_label == pred else "✗ 错误"
    if true_label != pred:
        all_correct = False
    print(f"{x_str:<12} {true_label:<10} {prob:<12.4f} {pred:<8} {status}")

print("-" * 50)
if all_correct:
    print("全部正确！通过梯度下降，模型自动学会了 XOR！")
else:
    print("有错误，可能需要更多训练轮次或调整学习率。")

---
## 7. 翻译词典

| 生活直觉 | 深度学习术语 | 英文 |
|----------|-------------|------|
| GPS 说「还有 5 公里」 | 损失函数 | Loss Function |
| 直线距离的平方 | 均方误差 | Mean Squared Error (MSE) |
| 直线距离 | 平均绝对误差 | Mean Absolute Error (MAE) |
| 走错方向的概率 | 二元交叉熵 | Binary Cross-Entropy (BCE) |
| 蒙眼下山，感受坡度 | 梯度下降 | Gradient Descent |
| 每步迈多远 | 学习率 | Learning Rate |
| 球的惯性 | 动量 | Momentum |
| 自动调整步幅 | 自适应学习率 | Adaptive Learning Rate (Adam) |

---
## 8. 总结与预告

### 今日核心收获

1. **损失函数**：量化预测和真实之间的差距，模型的优化目标
2. **MSE/MAE/BCE**：不同任务用不同的损失函数
3. **梯度下降**：沿梯度反方向更新参数，一步步逼近最优解
4. **学习率**：最重要的超参数，太大发散，太小收敛慢
5. **Adam 优化器**：结合动量和自适应学习率，默认首选

### 明天预告

今天我们知道了「怎么衡量错误」和「怎么改正错误」。但梯度是怎么从输出层传回隐藏层的？

明天学习**反向传播**——用链式法则把误差信号逐层传回去，让每个权重都知道自己该承担多少责任。